# HW2: reproduce the best scored output

The 180,000-step scratch model with four-beam decoding scored **56.67% (544/960)**, rank 8/15 on 16 October at 08:23 (+03:00). CPU generation took 503.50 seconds. T4 timing for this decoding setting is deferred and has not been verified. The historical greedy T4 notebook is retained separately.

Run the CPU preparation first. Generation can stay on CPU; optionally select a GPU afterward. Every program is an exact model decode in the original 1,919-question test.parquet order. Results are saved to Drive before export. Run each phase separately.


In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Select CPU for file preparation'
from google.colab import drive
drive.mount('/content/drive', timeout_ms=180000)

In [ ]:
import os, sys, json, hashlib, shutil, tarfile, subprocess
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Keep GPU off during file checks'
os.environ['USE_TF']='0'; os.environ['USE_FLAX']='0'
root = Path('/content/drive/MyDrive/HSE_NLP_HW2')
assert Path('/content/drive').is_mount()
prepared = root/'prepared/inference-180000-beam4-v1'
prepared.mkdir(parents=True, exist_ok=True)
work = Path('/content/hw2-inference-prepared')
work.mkdir(exist_ok=True)
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(4*1024*1024), b''): h.update(block)
    return h.hexdigest()
source = root/'prepared/beam4-quality-180k-v1'
source_ready = json.loads((source/'ready.json').read_text())
assert source_ready['source_commit'] == 'd1aec6e4b490118d4877959bb59189f2bbfd33f3'
assert digest(source/'source.tar') == source_ready['source_sha256']
with tarfile.open(source/'source.tar') as z: z.extractall(work, filter='data')
os.chdir(work)
sys.path.insert(0, str(work))
subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
from homeworks.hw2.checkpoint_io import drive_preflight
from homeworks.hw2.data import fetch_parquet
drive_preflight(root)
parent = root/'runs/continue-60k-180k-b128-v1/step-180000'
manifest = json.loads((parent/'checkpoint.complete.json').read_text())
assert manifest['status'] == 'checkpoint_verified'
assert manifest['files']['model.safetensors'] == '0399a924eea6ef872aa0b403b77fc5e33d24909cc7690536880bc00b254362c7'
model_dir = work/'model'; model_dir.mkdir(exist_ok=True)
for name, expected in manifest['files'].items():
    assert Path(name).name == name
    if name != 'training_state.pt':
        assert digest(parent/name) == expected, name
        shutil.copyfile(parent/name, model_dir/name)
        assert digest(model_dir/name) == expected
shutil.copyfile(parent/'checkpoint.complete.json', model_dir/'checkpoint.complete.json')
test = fetch_parquet('data/test/test.parquet', work/'cache')
import pyarrow.parquet as pq
questions = pq.read_table(test,columns=['question'])['question'].to_pylist()
question_hash = hashlib.sha256(json.dumps(questions,ensure_ascii=False).encode()).hexdigest()
assert len(questions) == 1919
assert question_hash == '214746f8d4bc918b6fa1fde8ca3185a518a60637f239a2c45efa9a8143f612d3'
from transformers import PreTrainedTokenizerFast, T5ForConditionalGeneration
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_dir, local_files_only=True)
model = T5ForConditionalGeneration.from_pretrained(model_dir, local_files_only=True)
assert sum(p.numel() for p in model.parameters()) == 9967360
del model
bundle = Path('/content/inference-inputs.tar')
with tarfile.open(bundle,'w') as z:
    for folder in ['homeworks','model','cache']:
        z.add(work/folder, arcname=folder)
    z.add(work/'source_commit.txt',arcname='source_commit.txt')
target = prepared/'inputs.tar'
shutil.copyfile(bundle,target)
assert digest(target) == digest(bundle)
ready = {'source_commit':source_ready['source_commit'],'bundle_sha256':digest(target),
         'model_sha256':manifest['files']['model.safetensors'],'questions_sha256':question_hash,
         'rows':1919,'cpu_load_verified':True}
(prepared/'ready.json').write_text(json.dumps(ready,indent=2))
assert json.loads((prepared/'ready.json').read_text()) == ready
print('CPU_INFERENCE_READY',ready)

## Generation
Stay on CPU to reproduce the scored configuration without GPU cost. A GPU is optional after CPU_INFERENCE_READY. This cell restores the verified bundle, generates exact programs, saves them to Drive and releases its runtime. It does not download the training dataset.


In [ ]:
from google.colab import runtime
try:
    from google.colab import drive
    drive.mount('/content/drive', timeout_ms=180000)
    from pathlib import Path
    import os, sys, json, hashlib, shutil, tarfile, subprocess
    os.environ['USE_TF']='0'; os.environ['USE_FLAX']='0'
    root = Path('/content/drive/MyDrive/HSE_NLP_HW2')
    prepared = root/'prepared/inference-180000-beam4-v1'
    ready = json.loads((prepared/'ready.json').read_text())
    assert ready['cpu_load_verified'] and ready['rows'] == 1919
    work = Path('/content/hw2-inference-beam4'); work.mkdir(exist_ok=True)
    bundle = Path('/content/inference-inputs.tar')
    shutil.copyfile(prepared/'inputs.tar',bundle)
    assert hashlib.sha256(bundle.read_bytes()).hexdigest() == ready['bundle_sha256']
    with tarfile.open(bundle) as z: z.extractall(work,filter='data')
    os.chdir(work); sys.path.insert(0,str(work))
    subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
    from homeworks.hw2.checkpoint_io import sha256, drive_preflight, package_jsonl
    drive_preflight(root)
    model_dir = work/'model'
    assert sha256(model_dir/'model.safetensors') == ready['model_sha256']
    print('INPUTS_RESTORED')
    import torch, csv, datetime
    from zipfile import ZipFile, ZIP_DEFLATED
    from google.colab import runtime
    from homeworks.hw2.inference import generate_predictions
    from transformers import PreTrainedTokenizerFast
    output = root/'inference'/('step-180000-beam4-'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
    output.mkdir(parents=True,exist_ok=False)
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    if gpu == 'cpu': torch.set_num_threads(4)
    local = work/'results'; local.mkdir(exist_ok=True)
    count, seconds = generate_predictions(model_dir,local/'predictions.csv',work/'cache',
                                         batch_size=64,max_new_tokens=192,precision='fp32',num_beams=4)
    assert count == 1919
    provenance = json.loads((local/'predictions.provenance.json').read_text())
    assert provenance['model_sha256'] == ready['model_sha256']
    assert provenance['questions_sha256'] == ready['questions_sha256']
    assert provenance['handwritten_repairs'] == 0
    with (local/'predictions.csv').open(newline='') as f:
        programs = [r['code'] for r in csv.DictReader(f)]
    tokenizer = PreTrainedTokenizerFast.from_pretrained(model_dir)
    assert programs == tokenizer.batch_decode(provenance['selected_token_ids'],skip_special_tokens=True,clean_up_tokenization_spaces=False)
    jsonl_hash = package_jsonl(local/'predictions.csv',local/'predictions.jsonl')
    evidence = {'count':count,'generation_seconds':seconds,'gpu':gpu,'t4_within_180_seconds':seconds<=180 if 'T4' in gpu else None,
                'num_beams':4,'batch_size':64,
                'jsonl_sha256':jsonl_hash,'checkpoint':'continue-60k-180k-b128-v1/step-180000',
                'question_order':'official data/test/test.parquet','source_commit':ready['source_commit']}
    (local/'timing.json').write_text(json.dumps(evidence,indent=2))
    for name in ['predictions.csv','predictions.jsonl','predictions.provenance.json','timing.json']:
        shutil.copyfile(local/name,output/name)
        assert sha256(local/name) == sha256(output/name)
    # Build and export only from verified Drive copies.
    with ZipFile(output/'inference-results.zip','w',ZIP_DEFLATED) as z:
        for name in ['predictions.csv','predictions.jsonl','predictions.provenance.json','timing.json']:
            z.write(output/name,arcname=name)
    (output/'results.complete.json').write_text(json.dumps(evidence,indent=2))
    (prepared/'last-result.json').write_text(json.dumps({'path':str(output)}))
    print('SAVED_TO_DRIVE',str(output),evidence)
    print('T4 timing is deferred unless this run actually used a T4.')
finally:
    try:
        runtime.unassign()
    except Exception as error:
        print('Release failed: disconnect and delete runtime manually.',type(error).__name__)

## Export from Drive
Use the authenticated Drive CLI for downloads without a Colab session, or use the CPU export cell below. Verify the JSONL hash and saved token provenance before submission. The scored CPU output hash is `0b81f30dc998d4bcb9804ce64dd1b98891bb27f592c134f40c24c3dac2de0f75`; do not resubmit this same output. Different outputs need their own bot receipt.


In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Select CPU before exporting'
from google.colab import drive, files
drive.mount('/content/drive', timeout_ms=180000)
import json
root = Path('/content/drive/MyDrive/HSE_NLP_HW2')
output = Path(json.loads((root/'prepared/inference-180000-beam4-v1/last-result.json').read_text())['path'])
assert output.is_relative_to(root/'inference')
print(json.loads((output/'results.complete.json').read_text()))
files.download(str(output/'inference-results.zip'))